# 01 — LangChain Integration

This notebook covers your first interaction with an LLM via code — from a simple "hello world" call to structured outputs, system prompts, and conversational memory.

## Setup

Load environment variables and initialize the Gemini model.

In [30]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pprint import pprint

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
)
print("Model ready!")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Model ready!


## First LLM Call & Model Parameters

Let's make our first LLM call and examine what comes back.

The `invoke()` method returns an `AIMessage` object. The main fields:
- `.content` — the actual text response
- `.usage_metadata` — dict with token counts


### Call invoke and inspect the output


In [52]:
response = llm.invoke("Apa ibukota Perancis?")
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [53]:
pprint(response.text)

'Ibukota Perancis adalah **Paris**.'


In [54]:
pprint(response.usage_metadata)

{'input_token_details': {'cache_read': 0},
 'input_tokens': 8,
 'output_tokens': 9,
 'total_tokens': 17}


### Compare Temperature Settings

The `temperature` parameter controls randomness:
- **0** = deterministic, always picks the most likely token
- **1** = creative, can produce more varied outputs

In [56]:

llm_creative = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=1,
)
llm_strict = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

question = "Berikan 3 sarapan yang bisa di buat dari telur."
print("Question: ", question)
print("=== Temperature = 1 ===")
response = llm_creative.invoke(question)
print(response.text)

print("\n=== Temperature = 0 === ")
response = llm_strict.invoke(question)
print(response.text)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Question:  Berikan 3 sarapan yang bisa di buat dari telur.
=== Temperature = 1 ===
Telur adalah bahan makanan yang sangat serbaguna untuk sarapan. Berikut adalah 3 ide sarapan dari telur dengan tingkat kesulitan yang berbeda:

### 1. Telur Orak-Arik Creamy (*Creamy Scrambled Eggs*)
Ini adalah menu klasik yang cepat dan sangat mengenyangkan.
*   **Cara buat:** Kocok 2 butir telur dengan sedikit susu cair, garam, dan lada hitam. Panaskan sedikit mentega di teflon dengan api kecil. Masukkan telur, lalu aduk perlahan secara konstan hingga lembut dan matang namun tetap lembap (*creamy*).
*   **Saran penyajian:** Sajikan di atas selembar roti panggang (*toast*).

### 2. Telur Ceplok Bumbu Kecap
Cocok untuk Anda yang suka sarapan dengan cita rasa gurih-manis khas Indonesia.
*   **Cara buat:** Ceplok telur seperti biasa hingga tingkat kematangan yang diinginkan. Di wajan yang sama (atau setelah telur diangkat), tumis sedikit bawang putih dan bawang merah cincang, tambahkan kecap manis, sedikit

The higher the temperature will resulted in more verbose and creative response,
while temperature=0 stays concise and factual.


### System Prompts

LLM models also accept input in the form of chat messages.
In LangChain, each chat message is categorized into three distinct roles:

* System – Provides initial instructions (typically used only in the first message).
* Human – Represents user input or prompts.
* AI – Represents the LLM's response.

In [57]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Anda adalah seorang bajak laut yang mengerti mengenai komputer. Jawab pertanyaan dengan bergaya bajak laut, namun jawab dengan singkat saja."),
    ("human", "Bagaimana memasak telur rebus?"),
])

messages = prompt.format_messages()
response = llm.invoke(messages)
print(response.text)


Lempar telur ke dalam panci air mendidih, kawan! Tunggu 6 menit jika ingin kuningnya cair, atau 10 menit jika ingin matang sempurna. Angkat, siram air dingin, lalu kupas kulitnya! Arrr!


### Structured JSON Output

Now a practical use case: use the system prompt to enforce JSON output format. This is essential when you need machine-parseable responses.

In [58]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Jawab pertanyaan dalam bentuk struktur JSON."),
    ("human", "Berikan 3 kota terpadat beserta nama negaranya"),
])

messages = prompt.format_messages()
response = llm.invoke(messages)
print(response.text)


```json
{
  "kota_terpadat": [
    {
      "peringkat": 1,
      "nama_kota": "Tokyo",
      "negara": "Jepang"
    },
    {
      "peringkat": 2,
      "nama_kota": "Delhi",
      "negara": "India"
    },
    {
      "peringkat": 3,
      "nama_kota": "Shanghai",
      "negara": "Tiongkok"
    }
  ]
}
```


## Out-of-Scope Refusal

System prompts define the persona and behavioral boundaries.

In [60]:
system_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Anda adalah chef masakan indonesia yang hanya boleh menjawab pertanyaan seputar masakan Indonesia. Tolak pertanyaan diluar cakupan ini."
    )),
    ("human", "{question}"),
])

chain = system_prompt | llm

print("=== IN-SCOPE ===")
print(chain.invoke({"question": "Bagaimana cara membuat rendang?"}).text)

=== IN-SCOPE ===
Tentu, mari kita membuat rendang daging sapi khas Minang yang autentik. Kunci dari rendang yang lezat adalah proses memasak perlahan dengan santan hingga bumbu meresap dan mengental menjadi dedak (bumbu kering).

Berikut adalah resepnya:

### Bahan-bahan:
*   **Daging:** 1 kg daging sapi (pilih bagian paha agar tidak mudah hancur), potong sesuai selera.
*   **Santan:** 1,5 - 2 liter santan kental dari 3 butir kelapa tua.
*   **Bumbu Halus:**
    *   100 gr cabai merah keriting (sesuaikan pedasnya).
    *   10 siung bawang merah.
    *   5 siung bawang putih.
    *   3 cm jahe.
    *   3 cm lengkuas.
    *   1 sdm ketumbar bubuk.
    *   1 sdt merica bubuk.
    *   1/2 sdt jintan.
*   **Bumbu Aromatik:**
    *   2 batang serai, memarkan.
    *   3 lembar daun kunyit, ikat simpul.
    *   5 lembar daun jeruk.
    *   3 lembar daun salam.
    *   1 buah asam kandis.
*   **Penyedap:** Garam secukupnya.

### Cara Membuat:
1.  **Masak Santan:** Masukkan santan ke dalam kuali

In [61]:

print("\n=== OUT-OF-SCOPE ===")
print(chain.invoke({"question": "Apa toping pizza terbaik?"}).text)


=== OUT-OF-SCOPE ===
Mohon maaf, saya hanya bisa menjawab pertanyaan seputar masakan Indonesia. Jika Anda ingin tahu tentang resep, teknik memasak, atau sejarah hidangan khas Nusantara, silakan bertanya kepada saya. Ada yang bisa saya bantu terkait masakan Indonesia?


## Chat Memory

LLMs are **stateless** — they don't remember previous conversations. Every call is independent.

Let's demonstrate this problem, then fix it.

### Without Memory

Run this loop and try the sequence:
- Type: `My name is Budi`
- Type: `What is my name?`

The bot won't remember your name because no history is passed between turns.

In [46]:
print("Chat started! Try: 'My name is Budi' then 'What is my name?'")
print("Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        break

    prompt = ChatPromptTemplate.from_messages([("human", user_input)])
    response = llm.invoke(prompt.format_messages())
    print(f"Bot: {response.text}\n")

Chat started! Try: 'My name is Budi' then 'What is my name?'
Type 'exit' to quit.



You:  exit


### Manual Chat History

To make the LLM remember, we pass the full conversation history into each new prompt using a list of `("role", "content")` tuples.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content="What is the weather in Jakarta today?"),
    AIMessage(content="It is 32°C and partly cloudy in Jakarta."),
    HumanMessage(content="Should I bring an umbrella?"),
    AIMessage(content="There is a 30% chance of rain, so it might be safe without one, but a small umbrella wouldn't hurt."),
]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly weather assistant."),
    *chat_history,
    ("human", "What about tomorrow?"),
])

response = (prompt | llm).invoke({})
print(response.content)

### With Memory

Now the same name-remembering test, but with history accumulated between turns.

In [50]:
print("Chat started! Try: 'My name is Budi' then 'What is my name?'")
print("Type 'exit' to quit.\n")

chat_history = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        break

    chat_history.append(("human", user_input))
    prompt = ChatPromptTemplate.from_messages(chat_history)
    response = llm.invoke(prompt.format_messages())
    print(f"Bot: {response.text}\n")
    chat_history.append(("ai", response.text))

Chat started! Try: 'My name is Budi' then 'What is my name?'
Type 'exit' to quit.



You:  my name is budi


Bot: Nice to meet you, Budi! How are you doing today? Is there anything I can help you with?



You:  what's my name?


Bot: Your name is Budi!



You:  quit


In [51]:
chat_history

[('human', 'my name is budi'),
 ('ai',
  'Nice to meet you, Budi! How are you doing today? Is there anything I can help you with?'),
 ('human', "what's my name?"),
 ('ai', 'Your name is Budi!')]